# 06 — Hybrid (Stage E, Arm 3)

The hybrid is the Stage D CNN (EfficientNet-B0, fine-tuned) with the ten classical features at
224 px (log1p + standardised on train) as extra inputs to both heads. It is trained end to end with
the Stage D recipe unchanged, so CNN vs hybrid differ only in those extra inputs. Both arms were also
fine-tuned with seeds 43 and 44 to measure run-to-run spread; the seed-42 runs are the arm entries.
This notebook trains nothing: it reads the saved val scores, training histories and the hybrid
checkpoint, and regenerates `reports/results_hybrid.md` and `reports/results_hybrid.csv`.

**Partitions:** trained on train; best epoch chosen on val; scored on val. The test partition is
never read.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json`,
`image_properties.csv`, `classical_features_224.csv`, `predictions/classical_gb_224_val.csv`,
`predictions/{cnn,hybrid}_finetune[_seed43|_seed44]_val.csv` and the matching
`cnn_runs/*/history.json`, plus `cnn_runs/hybrid_finetune/best.pt` (from `scripts/train_cnn.py`
and `scripts/train_cnn_seeds.py`). No GPU needed; runs in seconds.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "results_hybrid.md"
RESULTS_CSV = REPO_DIR / "reports" / "results_hybrid.csv"
PREDICTIONS_DIR = DATA_DIR / "predictions"
RUNS_DIR = DATA_DIR / "cnn_runs"

from src.data.splits import MODELLED_FLAWS, get_splits
from src.eda import image_props as ip
from src.eda import labels as lb
from src.eda import report as rp
from src.features.classical import FEATURE_NAMES
from src.models import classical as cm
from src.models import metrics as mt
from src.models.cnn import extra_head_weights

SEEDS = (42, 43, 44)
CLASSICAL = "Classical GB @ 224"
CNN = "CNN fine-tuned"
HYBRID = "Hybrid"
ARMS = (CLASSICAL, CNN, HYBRID)   # entries: classical GB @ 224, CNN and hybrid at seed 42


def seed_suffix(seed):
    return "" if seed == 42 else f"_seed{seed}"


RUNS = {(arm, seed): (f"{prefix}finetune{seed_suffix(seed)}", f"{pred}_finetune{seed_suffix(seed)}_val.csv")
        for arm, prefix, pred in ((CNN, "", "cnn"), (HYBRID, "hybrid_", "hybrid")) for seed in SEEDS}
FLAW_TYPE = {"BLR": "photometric", "DRK": "photometric", "BRT": "photometric",
             "FRM": "semantic", "OBS": "semantic", "ROT": "semantic", cm.UNREC: "recognisability"}
ENVIRONMENT = rp.environment_lines(
    gpu_note="not needed here: this notebook scores saved predictions; the training GPU is under "
             "Training runs",
    seed_note="arm entries use seed 42; seeds 43 and 44 measure run-to-run spread only")
pd.set_option("display.precision", 3)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(ENVIRONMENT))

## 1. Data
Targets come from the annotation records via `src/data/splits.py`. Each prediction file must hold
exactly the val images, with one score column per output.

In [ ]:
splits = get_splits(DATA_DIR)
targets = cm.build_targets(splits["val"])


def load_scores(filename):
    s = pd.read_csv(PREDICTIONS_DIR / filename, index_col=0)  # first column: image name
    s.index.name = "image"
    assert sorted(s.index) == sorted(targets.index), f"{filename}: image set differs from val"
    assert list(s.columns) == cm.OUTPUTS, f"{filename}: columns {list(s.columns)}"
    print(f"{filename:<32} {len(s):,} images, scores {s.values.min():.3f}-{s.values.max():.3f}")
    return s.loc[targets.index]


runs = {key: load_scores(filename) for key, (_, filename) in RUNS.items()}
scores = {CLASSICAL: load_scores("classical_gb_224_val.csv"), CNN: runs[CNN, 42], HYBRID: runs[HYBRID, 42]}
sections = []  # (heading, markdown) pairs collected for the report
sections.append(("Data", (
    f"Val {len(targets):,} images, scored by every run: the classical entry (GB @ 224) and the CNN "
    f"and hybrid at seeds {', '.join(map(str, SEEDS))}.")))

## 2. Training runs
Read from each run's `history.json`. All runs use the Stage D fine-tune recipe; the hybrid adds the
ten classical features as head inputs. Library versions are shown because Colab updated torch
between the Stage D CNN (seed 42) and the Stage E runs: the seed-43 and seed-44 pairs ran on the
same version, so they are the matched comparison.

In [ ]:
history, rows = {}, {}
for (arm, seed), (run_dir, _) in RUNS.items():
    h = json.loads((RUNS_DIR / run_dir / "history.json").read_text())
    history[arm, seed] = h
    epochs = pd.DataFrame(h["epochs"])
    torch_version = next((p.split()[1] for p in h["environment"]["libraries"].split(", ")
                          if p.startswith("torch ")), "?")
    rows[f"{arm}, seed {seed}"] = {
        "run": run_dir, "epochs run": len(epochs), "best epoch": h["best_epoch"],
        "best val macro AP": round(h["best_val_macro_ap"], 4), "s per epoch": round(epochs["seconds"].median()),
        "GPU": h["environment"]["gpu"], "torch": torch_version}
training = pd.DataFrame(rows).T
display(training)
c = history[HYBRID, 42]["config"]
sections.append(("Training runs", (
    f"`scripts/train_cnn.py --mode finetune [--hybrid] [--seed N]`: {c['backbone']}, batch "
    f"{c['batch_size']}, AdamW, lr {c['lr_head']:g} heads / {c['lr_backbone']:g} backbone, BCE with "
    f"pos_weight, mixed precision, early stop after {c['patience']} epochs without improvement in val "
    f"macro AP. Hybrid extra inputs: {c['extra_features']} (log1p + standardised on train).\n\n"
    + rp.df_to_markdown(training))))

## 3. Hybrid results on val (seed 42, the Stage E entry)
AP is primary and threshold-free; **read it against the base rate**. Gain = (AP - base rate) /
(1 - base rate). Precision, recall and F1 are at a fixed 0.5 threshold.

In [ ]:
def evaluate_with_gain(s):
    table = mt.evaluate(targets, s)
    table.insert(table.columns.get_loc("AP") + 1, "gain", mt.normalised_gain(table["AP"], table["base rate"]))
    table.loc[mt.MACRO, "gain"] = np.nan  # gain is read per output, not on the averaged row
    return table


results = {name: evaluate_with_gain(s) for name, s in scores.items()}
run_results = {key: evaluate_with_gain(s) for key, s in runs.items()}
display(results[HYBRID])

long = pd.concat({f"{arm}, seed {seed}": run_results[arm, seed] for arm, seed in RUNS},
                 names=["run", "output"])
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
long.reset_index().to_csv(RESULTS_CSV, index=False, float_format="%.4f")
sections.append(("Full table: Hybrid (seed 42)", rp.df_to_markdown(results[HYBRID].round(3))))

## 4. Three arms on val
Classical entry (GB @ 224), CNN entry and hybrid entry (seed 42), same val images, same metrics
code.

In [ ]:
rows = cm.OUTPUTS + [mt.MACRO]
arms = pd.DataFrame({"type": [FLAW_TYPE.get(o, "") for o in rows],
                     "base rate": results[CNN].loc[rows, "base rate"]}, index=rows)
for name in ARMS:
    arms[f"AP {name}"] = results[name].loc[rows, "AP"]
for name in ARMS:
    arms[f"gain {name}"] = results[name].loc[rows, "gain"]
display(arms)

by_type = arms.loc[MODELLED_FLAWS].groupby("type")[[f"gain {name}" for name in ARMS]].mean()
print("Mean gain by flaw type")
display(by_type)
f1 = pd.DataFrame({name: results[name].loc[rows, "F1"] for name in ARMS})
print("F1 @ 0.5")
display(f1)
sections.append(("Three arms on val", (
    "AP and gain per output (entries: classical GB @ 224, CNN and hybrid at seed 42).\n\n"
    + rp.df_to_markdown(arms.round(3)) + "\n\nMean gain by flaw type:\n\n"
    + rp.df_to_markdown(by_type.round(3)) + "\n\nF1 at threshold 0.5:\n\n"
    + rp.df_to_markdown(f1.round(3)))))

## 5. Is the hybrid difference larger than seed noise?
Macro AP per seed for both arms, then the paired difference (hybrid - CNN, same seed) per output.
A difference is consistent only if it has the same sign in every seed; with three seeds a
same-sign pattern still arises by chance one time in four, so small consistent differences are
reported as indicative.

In [ ]:
macro = pd.DataFrame({arm: [run_results[arm, s].loc[mt.MACRO, "AP"] for s in SEEDS] for arm in (CNN, HYBRID)},
                     index=[f"seed {s}" for s in SEEDS])
macro["hybrid - CNN"] = macro[HYBRID] - macro[CNN]
summary = macro.agg(["mean", "min", "max"])
macro_table = pd.concat([macro, summary])
display(macro_table)

paired = pd.DataFrame({f"seed {s}": run_results[HYBRID, s].loc[rows, "AP"] - run_results[CNN, s].loc[rows, "AP"]
                       for s in SEEDS})
paired["mean"] = paired.mean(axis=1)
paired["seeds hybrid ahead"] = (paired[[f"seed {s}" for s in SEEDS]] > 0).sum(axis=1)
paired["seeds hybrid behind"] = (paired[[f"seed {s}" for s in SEEDS]] < 0).sum(axis=1)
print("Paired AP difference, hybrid - CNN")
display(paired)

spread = pd.DataFrame({
    f"{arm} {stat}": getattr(pd.concat([run_results[arm, s].loc[rows, "AP"] for s in SEEDS], axis=1), stat)(axis=1)
    for arm in (CNN, HYBRID) for stat in ("mean", "min", "max")})
print("AP across seeds")
display(spread)
sections.append(("Seed spread and paired differences (val)", (
    "Macro AP over the six flaws by seed:\n\n" + rp.df_to_markdown(macro_table.round(4))
    + "\n\nPaired AP difference, hybrid - CNN, same seed:\n\n" + rp.df_to_markdown(paired.round(4))
    + "\n\nAP across seeds:\n\n" + rp.df_to_markdown(spread.round(3)))))

## 6. What the hybrid uses from the classical features
Head weights on the ten standardised features (seed 42 checkpoint): sign is the direction, magnitude
the strength. The last row compares the size of the classical part of each logit with the whole
logit over the val images (standard deviation of the classical contribution / standard deviation
of the full logit, recovered from the saved scores). A small ratio means the heads lean on the CNN
embedding and use the classical features as a correction.

In [ ]:
state = torch.load(RUNS_DIR / "hybrid_finetune" / "best.pt", map_location="cpu", weights_only=True)
weights = extra_head_weights(state, FEATURE_NAMES)

# The scaled features exactly as the hybrid saw them: preprocessing refitted on train, as in training.
features = cm.load_features(DATA_DIR, 224)[FEATURE_NAMES]
train_names = [r["image"] for r in splits["train"]]
preprocessor = cm.feature_preprocessor().fit(features.loc[train_names])
val_features = pd.DataFrame(preprocessor.transform(features.loc[targets.index]),
                            index=targets.index, columns=FEATURE_NAMES)
classical_part = val_features @ weights
p = scores[HYBRID].clip(1e-6, 1 - 1e-6)
full_logit = np.log(p / (1 - p))
weights.loc["classical share of logit (std ratio)"] = classical_part.std() / full_logit.std()

display(weights.style.background_gradient(cmap="RdBu_r", axis=None, vmin=-1, vmax=1,
                                          subset=pd.IndexSlice[FEATURE_NAMES, :]).format("{:.2f}"))
sections.append(("Hybrid head weights on the classical features (seed 42)", rp.df_to_markdown(weights.round(2))))

## 7. ROT: content or shape?
The Stage C bars: ROT AP above the shape-only reference on all val images, and above chance within
EXIF orientation 1, where raw shape carries no signal.

In [ ]:
props = ip.load_properties(DATA_DIR / "image_properties.csv")
val_props = props[props["image"].isin(targets.index)]
shape_ref = ip.rot_shortcut_scores(val_props, lb.vote_matrix(splits["val"]))
display(shape_ref)

orientation = props.set_index("image").loc[targets.index, "orientation"]
rot_rows = {}
for name, s in scores.items():
    row = {f"AP, all (n={len(targets)}, base {targets['ROT'].mean():.3f})":
           mt.average_precision(targets["ROT"], s["ROT"])}
    for tag in (1, 6):
        idx = orientation.index[orientation == tag]
        y = targets.loc[idx, "ROT"]
        ap = mt.average_precision(y, s.loc[idx, "ROT"])
        row[f"AP, orientation {tag} (n={len(idx)}, base {y.mean():.3f})"] = ap
        row[f"gain, orientation {tag}"] = mt.normalised_gain(ap, y.mean())
    rot_rows[name] = row
rot_by_orientation = pd.DataFrame(rot_rows).T
display(rot_by_orientation)
sections.append(("ROT: content or shape? (val)", (
    "Geometry-only cues scored as ROT predictors on val:\n\n" + rp.df_to_markdown(shape_ref)
    + "\n\nROT AP by EXIF orientation:\n\n" + rp.df_to_markdown(rot_by_orientation.round(3)))))

## 8. BLR on natively small images
BLR AP split by native long edge, as in Stages C and D.

In [ ]:
native_long = props.set_index("image").loc[targets.index, ["width", "height"]].max(axis=1)
blr_rows = {}
for name, s in scores.items():
    row = {}
    for label, mask in (("native < 512", native_long < 512), ("native >= 512", native_long >= 512)):
        y = targets.loc[mask, "BLR"]
        row[f"{label} (n={int(mask.sum())}, base {y.mean():.3f})"] = mt.average_precision(y, s.loc[y.index, "BLR"])
    blr_rows[name] = row
blr_by_size = pd.DataFrame(blr_rows).T
display(blr_by_size)
sections.append(("BLR by native resolution (val)", rp.df_to_markdown(blr_by_size.round(3))))

## 9. Single-flaw check
AP for each flaw on images carrying no other modelled flaw (positives: that flaw alone; negatives:
no flaw). In Stage D the classical arm matched the CNN on isolated DRK and BRT; this shows whether
the hybrid closes or keeps that pattern. Small positive counts make some rows indicative only.

In [ ]:
single = {name: mt.single_flaw_ap(targets, s) for name, s in scores.items()}
first = next(iter(single.values()))
single_table = first[["n", "positives (flaw alone)", "base rate"]].copy()
for name, t in single.items():
    single_table[f"AP {name}"] = t["AP"]
for name in ARMS:
    single_table[f"gain {name}"] = mt.normalised_gain(single_table[f"AP {name}"], single_table["base rate"])
display(single_table)
sections.append(("Single-flaw check (val)", rp.df_to_markdown(single_table.round(3))))

## 10. Write the report
The findings block in `reports/results_hybrid.md` is hand-written after reading the results and is
preserved on every rerun.

In [ ]:
out = rp.write_report(REPORT_PATH, sections, title="Stage E: Hybrid (Arm 3)",
                      environment=ENVIRONMENT, findings_heading="Findings (does the hybrid add anything?)")
print("Report written to", out)
print("Results table written to", RESULTS_CSV)